In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.multioutput import MultiOutputRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import joblib

print("Loading and preparing data...")
# Assume 'file.csv' is in the working directory
df = pd.read_csv('file.csv')

# Simulate the filtering you mentioned (keep clear/cloudy/rain/fog variants)
# You can adjust this list based on your exact 'df_filtered' logic
valid_weather = [
    'Mainly Clear', 
    'Mostly Cloudy', 
    'Cloudy', 
    'Clear', 
    'Rain', 
    'Rain Showers', 
    'Fog', 
    'Rain,Fog', 
    'Drizzle,Fog', 
    'Drizzle', 
    'Thunderstorms,Rain Showers', 
    'Haze', 
    'Thunderstorms,Rain Showers,Fog', 
    'Thunderstorms,Rain', 
    'Rain,Haze', 
    'Thunderstorms', 
    'Thunderstorms,Heavy Rain Showers', 
    'Thunderstorms,Moderate Rain Showers,Fog', 
    'Thunderstorms,Rain,Fog', 
    'Moderate Rain,Fog', 
    'Rain Showers,Fog'
]

df = df[df['Weather'].isin(valid_weather)].copy()

# Standardize time and reindex to fix gaps caused by filtering
df['Date/Time'] = pd.to_datetime(df['Date/Time'])
df = df.set_index('Date/Time')
df = df.sort_index()
# Resample to 1H to ensure strict chronological spacing, interpolate missing
df = df.resample('1H').interpolate(method='time')

print("Engineering features...")
# 1. Domain Features
df['Dew_Point_Deficit'] = df['Temp_C'] - df['Dew Point Temp_C']
# Proxy Index: High RH, low wind, low dew deficit = High fog risk
df['Fog_Stability_Index'] = df['Rel Hum_%'] / (1 + df['Wind Speed_km/h'] + df['Dew_Point_Deficit'])

# 2. Cyclical Time Features
df['Hour_Sin'] = np.sin(2 * np.pi * df.index.hour / 24)
df['Hour_Cos'] = np.cos(2 * np.pi * df.index.hour / 24)
df['Month_Sin'] = np.sin(2 * np.pi * df.index.month / 12)
df['Month_Cos'] = np.cos(2 * np.pi * df.index.month / 12)

# 3. Lags and Rolling Features
target_cols = ['Temp_C', 'Rel Hum_%', 'Visibility_km', 'Press_kPa']
for col in target_cols:
    df[f'{col}_lag1'] = df[col].shift(1)
    df[f'{col}_lag3'] = df[col].shift(3)
    df[f'{col}_roll_mean_3h'] = df[col].rolling(window=3).mean()
    df[f'{col}_roll_std_3h'] = df[col].rolling(window=3).std()
    df[f'{col}_roll_mean_6h'] = df[col].rolling(window=6).mean()

# 4. Shifted Targets (t+1, t+2, t+3)
df['Target_t1'] = df['Visibility_km'].shift(-1)
df['Target_t2'] = df['Visibility_km'].shift(-2)
df['Target_t3'] = df['Visibility_km'].shift(-3)

# Drop NaNs created by shifts/rolling
df.dropna(inplace=True)

# Define Features and Targets
target_names = ['Target_t1', 'Target_t2', 'Target_t3']
exclude_cols = ['Dew Point Temp_C', 'Weather'] + target_names 
features = [c for c in df.columns if c not in exclude_cols]

X = df[features]
y = df[target_names]

# Chronological Train/Test Split (80/20)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]
df_train = df.iloc[:split_idx]

# Calculate dynamic risk percentiles from Training target distribution
p33 = np.percentile(df_train['Visibility_km'], 33)
p66 = np.percentile(df_train['Visibility_km'], 66)
print(f"Risk thresholds based on training data - High Risk: <{p33:.1f}km, Low Risk: >{p66:.1f}km")

print("\nEvaluating Naive Persistence Baseline...")
# Persistence: predict the current visibility for all future horizons
persistence_preds = np.column_stack([
    X_test['Visibility_km'].values,
    X_test['Visibility_km'].values,
    X_test['Visibility_km'].values
])

print("Training Multi-Output XGBoost...")
# Using MultiOutputRegressor to train a separate XGBoost per horizon automatically
base_xgb = XGBRegressor(n_estimators=100, max_depth=5, learning_rate=0.1, random_state=42)
model = MultiOutputRegressor(base_xgb)
model.fit(X_train, y_train)

xgb_preds = model.predict(X_test)

print("\n--- PERFORMANCE METRICS ---")
print(f"{'Horizon':<10} | {'Model':<15} | {'RMSE':<6} | {'MAE':<6} | {'R²':<6}")
print("-" * 55)

for i, horizon in enumerate(['t+1', 't+2', 't+3']):
    # Persistence Metrics
    p_rmse = np.sqrt(mean_squared_error(y_test.iloc[:, i], persistence_preds[:, i]))
    p_mae = mean_absolute_error(y_test.iloc[:, i], persistence_preds[:, i])
    p_r2 = r2_score(y_test.iloc[:, i], persistence_preds[:, i])
    print(f"{horizon:<10} | {'Persistence':<15} | {p_rmse:.2f} | {p_mae:.2f} | {p_r2:+.2f}")
    
    # XGBoost Metrics
    m_rmse = np.sqrt(mean_squared_error(y_test.iloc[:, i], xgb_preds[:, i]))
    m_mae = mean_absolute_error(y_test.iloc[:, i], xgb_preds[:, i])
    m_r2 = r2_score(y_test.iloc[:, i], xgb_preds[:, i])
    print(f"{horizon:<10} | {'XGBoost':<15} | {m_rmse:.2f} | {m_mae:.2f} | {m_r2:+.2f}")
    print("-" * 55)

print("\nSaving models and artifacts for Streamlit...")
joblib.dump(model, 'xgb_model.joblib')
joblib.dump({'features': features, 'p33': p33, 'p66': p66}, 'artifacts.joblib')
# Save test set to allow interactive selection in Streamlit
X_test.to_csv('test_features.csv')
print("Done! You can now run: streamlit run streamlit_app.py")

Loading and preparing data...
Engineering features...
Risk thresholds based on training data - High Risk: <24.1km, Low Risk: >25.0km

Evaluating Naive Persistence Baseline...
Training Multi-Output XGBoost...


C:\Users\bajor\AppData\Local\Temp\ipykernel_26900\332766022.py:46: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  df = df.resample('1H').interpolate(method='time')
C:\Users\bajor\AppData\Local\Temp\ipykernel_26900\332766022.py:46: FutureWarning: DataFrame.interpolate with object dtype is deprecated and will raise in a future version. Call obj.infer_objects(copy=False) before interpolating instead.
  df = df.resample('1H').interpolate(method='time')



--- PERFORMANCE METRICS ---
Horizon    | Model           | RMSE   | MAE    | R²    
-------------------------------------------------------
t+1        | Persistence     | 5.36 | 1.72 | +0.67
t+1        | XGBoost         | 5.61 | 3.47 | +0.64
-------------------------------------------------------
t+2        | Persistence     | 7.34 | 2.94 | +0.38
t+2        | XGBoost         | 6.70 | 4.18 | +0.48
-------------------------------------------------------
t+3        | Persistence     | 8.70 | 3.98 | +0.13
t+3        | XGBoost         | 7.70 | 5.12 | +0.32
-------------------------------------------------------

Saving models and artifacts for Streamlit...
Done! You can now run: streamlit run streamlit_app.py


In [2]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, f1_score, recall_score, precision_score
from xgboost import XGBRegressor
from sklearn.multioutput import MultiOutputRegressor
import joblib

print("Loading dataset...")
df = pd.read_csv('file.csv')

# Filter relevant weather conditions
valid_weather = ['Clear', 'Cloudy', 'Mainly Clear', 'Mostly Cloudy', 'Fog', 
                 'Rain', 'Rain,Fog', 'Drizzle,Fog', 'Drizzle', 'Thunderstorms', 'Moderate Fog']
df = df[df['Weather'].isin(valid_weather)].copy()

# Fix deprecation: parse datetime and resample using lowercase '1h'
df['Date/Time'] = pd.to_datetime(df['Date/Time'])
df = df.set_index('Date/Time').sort_index()
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].resample('1h').interpolate(method='time')
df['Weather'] = df['Weather'].ffill()

print("Engineering features...")
# Domain Features
df['Dew_Point_Deficit'] = df['Temp_C'] - df['Dew Point Temp_C']
df['Fog_Stability_Index'] = df['Rel Hum_%'] / (1.0 + df['Wind Speed_km/h'] + np.maximum(df['Dew_Point_Deficit'], 0))
df['Temp_Drop_1h'] = df['Temp_C'] - df['Temp_C'].shift(1)
df['Press_Drop_3h'] = df['Press_kPa'] - df['Press_kPa'].shift(3)

# Cyclical Temporal Encodings
df['Hour_Sin'] = np.sin(2 * np.pi * df.index.hour / 24)
df['Hour_Cos'] = np.cos(2 * np.pi * df.index.hour / 24)
df['Month_Sin'] = np.sin(2 * np.pi * df.index.month / 12)
df['Month_Cos'] = np.cos(2 * np.pi * df.index.month / 12)

# Lags & Rolling Statistics
target_cols = ['Temp_C', 'Rel Hum_%', 'Visibility_km', 'Press_kPa']
for col in target_cols:
    df[f'{col}_lag1'] = df[col].shift(1)
    df[f'{col}_lag2'] = df[col].shift(2)
    df[f'{col}_lag3'] = df[col].shift(3)
    df[f'{col}_roll_mean_3h'] = df[col].rolling(window=3).mean()
    df[f'{col}_roll_std_3h'] = df[col].rolling(window=3).std()
    df[f'{col}_roll_mean_6h'] = df[col].rolling(window=6).mean()

# Multi-Horizon Targets
df['Target_t1'] = df['Visibility_km'].shift(-1)
df['Target_t2'] = df['Visibility_km'].shift(-2)
df['Target_t3'] = df['Visibility_km'].shift(-3)

df.dropna(inplace=True)

target_names = ['Target_t1', 'Target_t2', 'Target_t3']
exclude_cols = ['Dew Point Temp_C', 'Weather'] + target_names 
features = [c for c in df.columns if c not in exclude_cols]

X = df[features]
y = df[target_names]

# Chronological 80/20 Train/Test Split
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# Define Domain Thresholds (km) for FogGuard Dynamic Control
FOG_THRESHOLD_CRITICAL = 3.0  # < 3km: Severe hazard
FOG_THRESHOLD_ADVISORY = 10.0 # 3-10km: Advisory hazard

print("Training Multi-Output XGBoost Regressor...")
base_xgb = XGBRegressor(
    n_estimators=180,
    max_depth=5,
    learning_rate=0.06,
    subsample=0.85,
    colsample_bytree=0.85,
    random_state=42
)
model = MultiOutputRegressor(base_xgb)
model.fit(X_train, y_train)

# Predictions
xgb_preds = model.predict(X_test)
persistence_preds = np.column_stack([
    X_test['Visibility_km'].values,
    X_test['Visibility_km'].values,
    X_test['Visibility_km'].values
])

print("\n" + "="*68)
print(f"{'Horizon':<8} | {'Model':<12} | {'RMSE':<6} | {'MAE':<6} | {'R²':<6} | {'Fog F1':<6} | {'Recall':<6}")
print("="*68)

for i, horizon in enumerate(['t+1', 't+2', 't+3']):
    y_true = y_test.iloc[:, i].values
    y_pred_xgb = xgb_preds[:, i]
    y_pred_pers = persistence_preds[:, i]
    
    # Regression
    p_rmse = np.sqrt(mean_squared_error(y_true, y_pred_pers))
    p_mae = mean_absolute_error(y_true, y_pred_pers)
    p_r2 = r2_score(y_true, y_pred_pers)
    
    m_rmse = np.sqrt(mean_squared_error(y_true, y_pred_xgb))
    m_mae = mean_absolute_error(y_true, y_pred_xgb)
    m_r2 = r2_score(y_true, y_pred_xgb)
    
    # Operational Safety Event Metrics (Visibility < 10 km)
    true_event = (y_true <= FOG_THRESHOLD_ADVISORY).astype(int)
    pred_event_xgb = (y_pred_xgb <= FOG_THRESHOLD_ADVISORY).astype(int)
    pred_event_pers = (y_pred_pers <= FOG_THRESHOLD_ADVISORY).astype(int)
    
    f1_xgb = f1_score(true_event, pred_event_xgb, zero_division=0)
    rec_xgb = recall_score(true_event, pred_event_xgb, zero_division=0)
    f1_pers = f1_score(true_event, pred_event_pers, zero_division=0)
    rec_pers = recall_score(true_event, pred_event_pers, zero_division=0)
    
    print(f"{horizon:<8} | {'Persistence':<12} | {p_rmse:.2f} | {p_mae:.2f} | {p_r2:+.2f} | {f1_pers:.2f}   | {rec_pers:.2f}")
    print(f"{horizon:<8} | {'XGBoost':<12} | {m_rmse:.2f} | {m_mae:.2f} | {m_r2:+.2f} | {f1_xgb:.2f}   | {rec_xgb:.2f}")
    print("-" * 68)

# Save artifacts
joblib.dump(model, 'xgb_model.joblib')
joblib.dump({
    'features': features,
    'crit_thresh': FOG_THRESHOLD_CRITICAL,
    'adv_thresh': FOG_THRESHOLD_ADVISORY
}, 'artifacts.joblib')
X_test.to_csv('test_features.csv')
print("\nPipeline artifacts successfully saved!")

Loading dataset...
Engineering features...
Training Multi-Output XGBoost Regressor...

Horizon  | Model        | RMSE   | MAE    | R²     | Fog F1 | Recall
t+1      | Persistence  | 5.49 | 1.79 | +0.66 | 0.78   | 0.78
t+1      | XGBoost      | 5.28 | 2.93 | +0.68 | 0.63   | 0.48
--------------------------------------------------------------------
t+2      | Persistence  | 7.53 | 3.05 | +0.36 | 0.67   | 0.67
t+2      | XGBoost      | 6.75 | 4.25 | +0.48 | 0.47   | 0.33
--------------------------------------------------------------------
t+3      | Persistence  | 8.93 | 4.13 | +0.09 | 0.58   | 0.58
t+3      | XGBoost      | 7.98 | 5.52 | +0.28 | 0.25   | 0.15
--------------------------------------------------------------------

Pipeline artifacts successfully saved!


In [3]:
"""
FogGuard training pipeline v2
------------------------------
Fixes the core problem from v1: XGBoost regression alone smooths predictions
toward the mean and MISSES fog events (t+3 recall was 0.15 vs 0.58 for naive
persistence). For a safety system, missed fog events are the metric that
matters most.

What changed vs v1:
  1. A 3-class classifier (CLEAR / ADVISORY / CRITICAL) per horizon, trained
     with balanced sample weights, drives the risk STATUS shown to the driver.
     This is what should decide the speed advisory, not the raw regression.
  2. Quantile regression (p10 / p50 / p90) per horizon gives an uncertainty
     band around the visibility forecast instead of one falsely-precise number.
  3. New features: 1h/3h visibility rate-of-change, and consecutive-hours-
     currently-in-fog (fog tends to persist, so "how long has it already been
     foggy" is a strong predictor of "will it still be foggy in 3h").
  4. TimeSeriesSplit cross-validation reported alongside the single held-out
     split, so your metrics aren't a fluke of one 80/20 cut.

Requires xgboost>=2.0 (for the native quantile objective). Check with:
    python -c "import xgboost; print(xgboost.__version__)"
If you're on an older version: pip install -U xgboost
"""

import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_squared_error, mean_absolute_error, r2_score,
    f1_score, recall_score, precision_score, classification_report,
)
from sklearn.model_selection import TimeSeriesSplit
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBRegressor, XGBClassifier
import joblib

# ---------------------------------------------------------------------------
# 0. Config
# ---------------------------------------------------------------------------
FOG_THRESHOLD_CRITICAL = 3.0   # km, < this: severe hazard
FOG_THRESHOLD_ADVISORY = 10.0  # km, between crit and this: advisory hazard
QUANTILES = [0.1, 0.5, 0.9]    # p10 / median / p90 forecast band
HORIZONS = [1, 2, 3]           # hours ahead
CLASS_LABELS = {0: "CLEAR", 1: "ADVISORY", 2: "CRITICAL"}

# ---------------------------------------------------------------------------
# 1. Load + clean
# ---------------------------------------------------------------------------
print("Loading dataset...")
df = pd.read_csv("file.csv")

valid_weather = [
    "Clear", "Cloudy", "Mainly Clear", "Mostly Cloudy", "Fog",
    "Rain", "Rain,Fog", "Drizzle,Fog", "Drizzle", "Thunderstorms",
    "Moderate Fog", "Rain Showers", "Haze", "Rain,Haze",
    "Thunderstorms,Rain", "Thunderstorms,Rain Showers",
    "Thunderstorms,Rain Showers,Fog", "Thunderstorms,Moderate Rain Showers,Fog",
    "Thunderstorms,Rain,Fog", "Moderate Rain,Fog", "Rain Showers,Fog",
    "Thunderstorms,Heavy Rain Showers",
]
df = df[df["Weather"].isin(valid_weather)].copy()

df["Date/Time"] = pd.to_datetime(df["Date/Time"])
df = df.set_index("Date/Time").sort_index()
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].resample("1h").interpolate(method="time")
df["Weather"] = df["Weather"].ffill()

# ---------------------------------------------------------------------------
# 2. Feature engineering
# ---------------------------------------------------------------------------
print("Engineering features...")

# -- Domain features (v1)
df["Dew_Point_Deficit"] = df["Temp_C"] - df["Dew Point Temp_C"]
df["Fog_Stability_Index"] = df["Rel Hum_%"] / (
    1.0 + df["Wind Speed_km/h"] + np.maximum(df["Dew_Point_Deficit"], 0)
)
df["Temp_Drop_1h"] = df["Temp_C"] - df["Temp_C"].shift(1)
df["Press_Drop_3h"] = df["Press_kPa"] - df["Press_kPa"].shift(3)

# -- NEW: visibility trend + fog persistence (strong signal fog will continue)
df["Vis_RoC_1h"] = df["Visibility_km"].diff(1)
df["Vis_RoC_3h"] = df["Visibility_km"].diff(3)
_fog_now = (df["Visibility_km"] <= FOG_THRESHOLD_ADVISORY).astype(int)
_run_id = (_fog_now != _fog_now.shift()).cumsum()
df["Consec_Fog_Hours"] = (_fog_now.groupby(_run_id).cumcount() + 1) * _fog_now

# -- Cyclical time encodings
df["Hour_Sin"] = np.sin(2 * np.pi * df.index.hour / 24)
df["Hour_Cos"] = np.cos(2 * np.pi * df.index.hour / 24)
df["Month_Sin"] = np.sin(2 * np.pi * df.index.month / 12)
df["Month_Cos"] = np.cos(2 * np.pi * df.index.month / 12)

# -- Lags & rolling stats
target_cols = ["Temp_C", "Rel Hum_%", "Visibility_km", "Press_kPa"]
for col in target_cols:
    df[f"{col}_lag1"] = df[col].shift(1)
    df[f"{col}_lag2"] = df[col].shift(2)
    df[f"{col}_lag3"] = df[col].shift(3)
    df[f"{col}_roll_mean_3h"] = df[col].rolling(3).mean()
    df[f"{col}_roll_std_3h"] = df[col].rolling(3).std()
    df[f"{col}_roll_mean_6h"] = df[col].rolling(6).mean()

# -- Multi-horizon regression targets
for h in HORIZONS:
    df[f"Target_t{h}"] = df["Visibility_km"].shift(-h)

df.dropna(inplace=True)

target_names = [f"Target_t{h}" for h in HORIZONS]
exclude_cols = ["Dew Point Temp_C", "Weather"] + target_names
features = [c for c in df.columns if c not in exclude_cols]

X = df[features]
y = df[target_names]

# Chronological 80/20 split (final held-out report)
split_idx = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_idx], X.iloc[split_idx:]
y_train, y_test = y.iloc[:split_idx], y.iloc[split_idx:]

# ---------------------------------------------------------------------------
# 3. Time-series cross-validation (robustness check, not just one lucky split)
# ---------------------------------------------------------------------------
print("\nRunning 5-fold TimeSeriesSplit CV (median regressor + classifier, t+1)...")
tscv = TimeSeriesSplit(n_splits=5)
cv_rmse, cv_f1 = [], []
y1 = df["Target_t1"]
y1_cls = np.where(y1 <= FOG_THRESHOLD_CRITICAL, 2, np.where(y1 <= FOG_THRESHOLD_ADVISORY, 1, 0))

for fold, (tr_idx, va_idx) in enumerate(tscv.split(X), 1):
    Xtr, Xva = X.iloc[tr_idx], X.iloc[va_idx]
    ytr, yva = y1.iloc[tr_idx], y1.iloc[va_idx]
    ytr_cls, yva_cls = y1_cls[tr_idx], y1_cls[va_idx]

    m = XGBRegressor(objective="reg:quantileerror", quantile_alpha=0.5,
                      n_estimators=150, max_depth=5, learning_rate=0.06,
                      subsample=0.85, colsample_bytree=0.85, random_state=42)
    m.fit(Xtr, ytr)
    rmse = np.sqrt(mean_squared_error(yva, m.predict(Xva)))
    cv_rmse.append(rmse)

    c = XGBClassifier(n_estimators=150, max_depth=5, learning_rate=0.07,
                       subsample=0.85, colsample_bytree=0.85, random_state=42,
                       eval_metric="mlogloss")
    w = compute_sample_weight("balanced", ytr_cls)
    c.fit(Xtr, ytr_cls, sample_weight=w)
    f1 = f1_score(yva_cls, c.predict(Xva), average="macro", zero_division=0)
    cv_f1.append(f1)
    print(f"  fold {fold}: RMSE={rmse:.2f}  macro-F1={f1:.2f}")

print(f"CV summary (t+1) -> RMSE {np.mean(cv_rmse):.2f} +/- {np.std(cv_rmse):.2f} | "
      f"macro-F1 {np.mean(cv_f1):.2f} +/- {np.std(cv_f1):.2f}")

# ---------------------------------------------------------------------------
# 4. Train final models per horizon (on the chronological train split)
# ---------------------------------------------------------------------------
reg_models = {h: {} for h in HORIZONS}
clf_models = {}

reg_kwargs = dict(n_estimators=180, max_depth=5, learning_rate=0.06,
                   subsample=0.85, colsample_bytree=0.85, random_state=42)
clf_kwargs = dict(n_estimators=200, max_depth=5, learning_rate=0.07,
                   subsample=0.85, colsample_bytree=0.85, random_state=42,
                   eval_metric="mlogloss")

print("\nTraining final per-horizon models...")
for h in HORIZONS:
    yh = y_train[f"Target_t{h}"]
    for q in QUANTILES:
        model = XGBRegressor(objective="reg:quantileerror", quantile_alpha=q, **reg_kwargs)
        model.fit(X_train, yh)
        reg_models[h][q] = model

    yh_cls = np.where(yh <= FOG_THRESHOLD_CRITICAL, 2,
              np.where(yh <= FOG_THRESHOLD_ADVISORY, 1, 0))
    clf = XGBClassifier(**clf_kwargs)
    weights = compute_sample_weight("balanced", yh_cls)
    clf.fit(X_train, yh_cls, sample_weight=weights)
    clf_models[h] = clf
    print(f"  horizon t+{h} done.")

# ---------------------------------------------------------------------------
# 5. Held-out evaluation vs persistence baseline
# ---------------------------------------------------------------------------
print("\n" + "=" * 78)
print(f"{'Horizon':<8} | {'Model':<12} | {'RMSE':<6} | {'MAE':<6} | {'R2':<6} | {'MacroF1':<8} | {'FogRecall':<9}")
print("=" * 78)

for h in HORIZONS:
    y_true = y_test[f"Target_t{h}"].values
    y_true_cls = np.where(y_true <= FOG_THRESHOLD_CRITICAL, 2,
                 np.where(y_true <= FOG_THRESHOLD_ADVISORY, 1, 0))

    # persistence baseline
    y_pers = X_test["Visibility_km"].values
    y_pers_cls = np.where(y_pers <= FOG_THRESHOLD_CRITICAL, 2,
                 np.where(y_pers <= FOG_THRESHOLD_ADVISORY, 1, 0))
    p_rmse = np.sqrt(mean_squared_error(y_true, y_pers))
    p_mae = mean_absolute_error(y_true, y_pers)
    p_r2 = r2_score(y_true, y_pers)
    p_f1 = f1_score(y_true_cls, y_pers_cls, average="macro", zero_division=0)
    p_rec = recall_score(y_true_cls, y_pers_cls, average="macro", zero_division=0)
    print(f"t+{h:<6} | {'Persistence':<12} | {p_rmse:.2f}   | {p_mae:.2f}   | {p_r2:+.2f}   | {p_f1:.2f}     | {p_rec:.2f}")

    # v2 model: median regressor for continuous error, classifier for risk class
    y_pred_med = reg_models[h][0.5].predict(X_test)
    y_pred_cls = clf_models[h].predict(X_test)
    m_rmse = np.sqrt(mean_squared_error(y_true, y_pred_med))
    m_mae = mean_absolute_error(y_true, y_pred_med)
    m_r2 = r2_score(y_true, y_pred_med)
    m_f1 = f1_score(y_true_cls, y_pred_cls, average="macro", zero_division=0)
    m_rec = recall_score(y_true_cls, y_pred_cls, average="macro", zero_division=0)
    print(f"t+{h:<6} | {'XGB v2':<12} | {m_rmse:.2f}   | {m_mae:.2f}   | {m_r2:+.2f}   | {m_f1:.2f}     | {m_rec:.2f}")
    print("-" * 78)

    if h == HORIZONS[-1]:
        print("\nDetailed classification report, furthest horizon (t+{}):".format(h))
        print(classification_report(y_true_cls, y_pred_cls,
                                     target_names=[CLASS_LABELS[i] for i in (0, 1, 2)],
                                     zero_division=0))

# ---------------------------------------------------------------------------
# 6. Save artifacts for Streamlit
# ---------------------------------------------------------------------------
joblib.dump({"reg": reg_models, "clf": clf_models}, "fog_models_v2.joblib")
joblib.dump({
    "features": features,
    "crit_thresh": FOG_THRESHOLD_CRITICAL,
    "adv_thresh": FOG_THRESHOLD_ADVISORY,
    "quantiles": QUANTILES,
    "horizons": HORIZONS,
    "class_labels": CLASS_LABELS,
}, "artifacts_v2.joblib")
X_test.to_csv("test_features_v2.csv")

print("\nSaved: fog_models_v2.joblib, artifacts_v2.joblib, test_features_v2.csv")
print("Run: streamlit run streamlit_app_v2.py")

Loading dataset...
Engineering features...

Running 5-fold TimeSeriesSplit CV (median regressor + classifier, t+1)...
  fold 1: RMSE=8.55  macro-F1=0.55
  fold 2: RMSE=6.86  macro-F1=0.40
  fold 3: RMSE=6.63  macro-F1=0.80
  fold 4: RMSE=6.05  macro-F1=0.56
  fold 5: RMSE=5.37  macro-F1=0.66
CV summary (t+1) -> RMSE 6.69 +/- 1.06 | macro-F1 0.59 +/- 0.13

Training final per-horizon models...
  horizon t+1 done.
  horizon t+2 done.
  horizon t+3 done.

Horizon  | Model        | RMSE   | MAE    | R2     | MacroF1  | FogRecall
t+1      | Persistence  | 5.43   | 1.78   | +0.66   | 0.74     | 0.74
t+1      | XGB v2       | 5.46   | 2.05   | +0.66   | 0.63     | 0.64
------------------------------------------------------------------------------
t+2      | Persistence  | 7.46   | 3.03   | +0.36   | 0.58     | 0.58
t+2      | XGB v2       | 6.70   | 2.94   | +0.49   | 0.51     | 0.54
------------------------------------------------------------------------------
t+3      | Persistence  | 8.83  